# Cadquery + ipywidget + dotbimpy

In [ ]:
from dotbimpy import *
from ipywidgets import interact
import plotly.graph_objects as go
import pyquaternion
import numpy as np
import cadquery
import uuid

In [ ]:
def cadquery_mesh_to_dotbim_mesh(cadquery_mesh, mesh_id):
    vertices, triangles = cadquery_mesh
    coordinates = []
    for i in vertices:
        coordinates.extend([i.x, i.y, i.z])
    indices = [item for sublist in triangles for item in sublist]
    return Mesh(mesh_id=mesh_id, coordinates=coordinates, indices=indices)

In [ ]:
def create_plotly_figure(file):
    geometries = []
    for i in file.elements:
        mesh = next((x for x in file.meshes if x.mesh_id == i.mesh_id), None)
        geometries.extend(convert_to_plotly_meshes_with_face_colors(mesh, element=i))

    layout = go.Layout(scene=dict(aspectmode="data"))
    figure = go.Figure(data=[], layout=layout)
    for i in geometries:
        figure.add_trace(i)

    return figure


def convert_to_plotly(mesh, element):
    color_hex = "#%02x%02x%02x" % (element.color.r, element.color.g, element.color.b)
    opacity = element.color.a / 255

    x, y, z = repack_mesh_vertices_to_xyz_lists(mesh, element)
    i, j, k = repack_mesh_indices_to_ijk_lists(mesh)

    return go.Mesh3d(
        x=x,
        y=y,
        z=z,
        i=i,
        j=j,
        k=k,
        color=color_hex,
        opacity=opacity,
        name=element.type,
        showscale=True,
    )


def convert_to_plotly_meshes_with_face_colors(mesh, element):
    if not element.check_if_has_face_colors():
        return [convert_to_plotly(mesh, element)]
    else:
        plotly_meshes = []
        face_colors_counter = 0
        indices_counter = 0
        while face_colors_counter < len(element.face_colors):
            color_hex = "#%02x%02x%02x" % (
                element.face_colors[face_colors_counter],
                element.face_colors[face_colors_counter + 1],
                element.face_colors[face_colors_counter + 2],
            )
            opacity = element.face_colors[face_colors_counter + 3] / 255

            i = [mesh.indices[indices_counter]]
            j = [mesh.indices[indices_counter + 1]]
            k = [mesh.indices[indices_counter + 2]]
            x, y, z = repack_mesh_vertices_to_xyz_lists(mesh, element)

            plotly_meshes.append(
                go.Mesh3d(
                    x=x,
                    y=y,
                    z=z,
                    i=i,
                    j=j,
                    k=k,
                    color=color_hex,
                    opacity=opacity,
                    name=element.type,
                    showscale=True,
                )
            )

            face_colors_counter += 4
            indices_counter += 3

        return plotly_meshes


def repack_mesh_indices_to_ijk_lists(mesh):
    i = []
    j = []
    k = []
    counter = 0
    while counter < len(mesh.indices):
        i.append(mesh.indices[counter])
        j.append(mesh.indices[counter + 1])
        k.append(mesh.indices[counter + 2])
        counter += 3

    return i, j, k


def repack_mesh_vertices_to_xyz_lists(mesh, element):

    x = []
    y = []
    z = []
    counter = 0
    while counter < len(mesh.coordinates):
        point = np.array(
            [
                mesh.coordinates[counter],
                mesh.coordinates[counter + 1],
                mesh.coordinates[counter + 2],
            ]
        )

        rotation = pyquaternion.Quaternion(
            a=element.rotation.qw,
            b=element.rotation.qx,
            c=element.rotation.qy,
            d=element.rotation.qz,
        )

        point_rotated = rotation.rotate(point)

        x.append(point_rotated[0] + element.vector.x)
        y.append(point_rotated[1] + element.vector.y)
        z.append(point_rotated[2] + element.vector.z)
        counter += 3

    return x, y, z

In [ ]:
def create_mesh_tube(plane, face, l, h, t, mesh_id):
    workplane = (
        cadquery.Workplane(plane)
        .circle(h)
        .extrude(l)
        .faces(face)
        .workplane()
        .circle(h - t)
        .cutThruAll()
    )
    mesh_cq = workplane.val().tessellate(0.1)
    mesh = cadquery_mesh_to_dotbim_mesh(mesh_cq, mesh_id)
    return mesh


def create_tube_element(vector, rotation, color, mesh_id, type_name):
    return Element(
        mesh_id=mesh_id,
        vector=vector,
        guid=str(uuid.uuid4()),
        info={"Material": "Steel"},
        rotation=rotation,
        type=type_name,
        color=color,
    )

In [ ]:
def f(length):
    meshes = []
    elements = []

    # Creating chords
    meshes.append(create_mesh_tube("YZ", ">X", length, 0.03, 0.01, 0))
    elements.append(
        create_tube_element(
            Vector(0, 0.25, 0.25),
            Rotation(0, 0, 0, 1),
            Color(0, 0, 255, 255),
            0,
            "Top Chord",
        )
    )
    elements.append(
        create_tube_element(
            Vector(0, -0.25, 0.25),
            Rotation(0, 0, 0, 1),
            Color(0, 0, 255, 255),
            0,
            "Top Chord",
        )
    )
    elements.append(
        create_tube_element(
            Vector(0, 0.25, -0.25),
            Rotation(0, 0, 0, 1),
            Color(0, 0, 255, 255),
            0,
            "Bottom Chord",
        )
    )
    elements.append(
        create_tube_element(
            Vector(0, -0.25, -0.25),
            Rotation(0, 0, 0, 1),
            Color(0, 0, 255, 255),
            0,
            "Bottom Chord",
        )
    )

    # Creating posts
    meshes.append(create_mesh_tube("XY", ">Z", 0.46, 0.02, 0.005, 1))
    elements.append(
        create_tube_element(
            Vector(0.25, 0.25, -0.23),
            Rotation(0, 0, 0, 1),
            Color(0, 255, 0, 255),
            1,
            "Post",
        )
    )
    elements.append(
        create_tube_element(
            Vector(0.25, -0.25, -0.23),
            Rotation(0, 0, 0, 1),
            Color(0, 255, 0, 255),
            1,
            "Post",
        )
    )
    elements.append(
        create_tube_element(
            Vector(length - 0.25, 0.25, -0.23),
            Rotation(0, 0, 0, 1),
            Color(0, 255, 0, 255),
            1,
            "Post",
        )
    )
    elements.append(
        create_tube_element(
            Vector(length - 0.25, -0.25, -0.23),
            Rotation(0, 0, 0, 1),
            Color(0, 255, 0, 255),
            1,
            "Post",
        )
    )

    # Creating girder
    meshes.append(create_mesh_tube("XZ", ">Y", 0.46, 0.02, 0.005, 2))
    elements.append(
        create_tube_element(
            Vector(0.25, 0.23, 0.25),
            Rotation(0, 0, 0, 1),
            Color(222, 49, 99, 255),
            2,
            "Girder",
        )
    )
    elements.append(
        create_tube_element(
            Vector(0.25, 0.23, -0.25),
            Rotation(0, 0, 0, 1),
            Color(222, 49, 99, 255),
            2,
            "Girder",
        )
    )
    elements.append(
        create_tube_element(
            Vector(length - 0.25, 0.23, 0.25),
            Rotation(0, 0, 0, 1),
            Color(222, 49, 99, 255),
            2,
            "Girder",
        )
    )
    elements.append(
        create_tube_element(
            Vector(length - 0.25, 0.23, -0.25),
            Rotation(0, 0, 0, 1),
            Color(222, 49, 99, 255),
            2,
            "Girder",
        )
    )

    # Creating webs
    length_to_fill = length - 0.5
    number_of_gaps = int(length_to_fill / 0.5)
    meshes.append(create_mesh_tube("YZ", ">X", 1.41421 * 0.5, 0.015, 0.005, 3))
    for i in range(number_of_gaps):
        if i % 2 == 0:
            elements.append(
                create_tube_element(
                    Vector(i * 0.5 + 0.25, 0.25, 0.5 / 2.0),
                    Rotation(0, 0.382683, 0, 0.92388),
                    Color(255, 165, 0, 255),
                    3,
                    "Web",
                )
            )
            elements.append(
                create_tube_element(
                    Vector(i * 0.5 + 0.25, -0.25, -0.5 / 2.0),
                    Rotation(0, -0.382683, 0, 0.92388),
                    Color(255, 165, 0, 255),
                    3,
                    "Web",
                )
            )
            elements.append(
                create_tube_element(
                    Vector(i * 0.5 + 0.25, 0.25, 0.5 / 2.0),
                    Rotation(0, 0, -0.382683, 0.92388),
                    Color(255, 165, 0, 255),
                    3,
                    "Web",
                )
            )
            elements.append(
                create_tube_element(
                    Vector(i * 0.5 + 0.25, -0.25, -0.5 / 2.0),
                    Rotation(0, 0, 0.382683, 0.92388),
                    Color(255, 165, 0, 255),
                    3,
                    "Web",
                )
            )
        else:
            elements.append(
                create_tube_element(
                    Vector(i * 0.5 + 0.25, 0.25, -0.5 / 2.0),
                    Rotation(0, -0.382683, 0, 0.92388),
                    Color(255, 165, 0, 255),
                    3,
                    "Web",
                )
            )
            elements.append(
                create_tube_element(
                    Vector(i * 0.5 + 0.25, -0.25, 0.5 / 2.0),
                    Rotation(0, 0.382683, 0, 0.92388),
                    Color(255, 165, 0, 255),
                    3,
                    "Web",
                )
            )
            elements.append(
                create_tube_element(
                    Vector(i * 0.5 + 0.25, -0.25, 0.5 / 2.0),
                    Rotation(0, 0, 0.382683, 0.92388),
                    Color(255, 165, 0, 255),
                    3,
                    "Web",
                )
            )
            elements.append(
                create_tube_element(
                    Vector(i * 0.5 + 0.25, 0.25, -0.5 / 2.0),
                    Rotation(0, 0, -0.382683, 0.92388),
                    Color(255, 165, 0, 255),
                    3,
                    "Web",
                )
            )

    # Creating webs on ends
    meshes.append(create_mesh_tube("XZ", ">Y", 1.41421 * 0.5, 0.015, 0.005, 4))
    elements.append(
        create_tube_element(
            Vector(0.25, 0.25, 0.5 / 2.0),
            Rotation(0.382683, 0, 0, 0.92388),
            Color(255, 105, 180, 255),
            4,
            "Web",
        )
    )
    elements.append(
        create_tube_element(
            Vector(length - 0.25, 0.25, -0.5 / 2.0),
            Rotation(-0.382683, 0, 0, 0.92388),
            Color(255, 105, 180, 255),
            4,
            "Web",
        )
    )

    file = File(
        schema_version="1.0.0",
        meshes=meshes,
        elements=elements,
        info={"Author": "John Doe"},
    )
    figure = create_plotly_figure(file)
    figure.show()
    # file.save("Truss.bim")

In [ ]:
interact(f, length=(2.0, 10.0, 1.00))